In [1]:
import json, random, re
from pathlib import Path
import pandas as pd
import numpy as np

IN_CSV  = "/content/cleaned_offside_data.csv"
OUT_DIR = Path("ft_out_offsides"); OUT_DIR.mkdir(parents=True, exist_ok=True)

num_re = re.compile(r"[-+]?\d+(?:[.,]\d+)?(?:[eE][-+]?\d+)?")
def parse_prr(x):
    if pd.isna(x): return np.nan
    m = num_re.search(str(x))
    if not m: return np.nan
    return float(m.group(0).replace(",", "."))

df = pd.read_csv(IN_CSV, low_memory=False)
req = {"drug_concept_name","condition_concept_name","PRR"}
missing = req - set(df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

df["PRR"] = df["PRR"].map(parse_prr)
df = df.dropna(subset=["drug_concept_name","condition_concept_name","PRR"])
df = df[df["PRR"] > 0]

df = (df.sort_values("PRR", ascending=False)
        .drop_duplicates(subset=["drug_concept_name","condition_concept_name"], keep="first"))

drugs = df["drug_concept_name"].drop_duplicates().tolist()
random.seed(42); random.shuffle(drugs)
n = len(drugs); n_train = int(0.8*n); n_val = int(0.1*n)
train_drugs = set(drugs[:n_train])
val_drugs   = set(drugs[n_train:n_train+n_val])
test_drugs  = set(drugs[n_train+n_val:])

PRR_POS  = 2.0
PER_POS  = 2
NEG_RATIO = 0.3

Q_TEMPLATES = [
    "What adverse event is associated with {d}? Respond with event and PRR.",
    "Which side effect is reported for {d}? Include PRR.",
    "For the drug {d}, what adverse condition has been reported? Provide PRR.",
    "List a likely adverse event for {d}. Show PRR."
]
NEG_TEMPLATES = [
    "Are there strong adverse signals for {d}? If not, say 'no strong signal'.",
    "Does {d} show a notable side effect? If weak, say it's weak."
]

splits = {"train": [], "val": [], "test": []}
for drug, g in df.groupby("drug_concept_name"):
    pos = g[g["PRR"] >= PRR_POS]
    neg = g[g["PRR"] <  PRR_POS]
    for _, r in pos.iterrows():
        for _ in range(PER_POS):
            inst = random.choice(Q_TEMPLATES).format(d=drug)
            out  = f"{r['condition_concept_name']} (PRR ≈ {round(r['PRR'],2)})."
            ex = {"instruction": inst, "input": drug, "output": out}
            bucket = "train" if drug in train_drugs else "val" if drug in val_drugs else "test"
            splits[bucket].append(ex)
    k = max(1, int(len(pos)*NEG_RATIO)) if len(pos) > 0 else min(len(neg), 1)
    if k > 0 and len(neg) > 0:
        take = neg.sample(n=min(k, len(neg)), random_state=42)
        for _, r in take.iterrows():
            inst = random.choice(NEG_TEMPLATES).format(d=drug)
            qual = "no strong signal" if r["PRR"] < 1.5 else "weak signal"
            out  = f"{qual} (PRR ≈ {round(r['PRR'],2)})."
            ex = {"instruction": inst, "input": drug, "output": out}
            bucket = "train" if drug in train_drugs else "val" if drug in val_drugs else "test"
            splits[bucket].append(ex)

for name in ["train","val","test"]:
    path = OUT_DIR / f"{name}.jsonl"
    with open(path, "w", encoding="utf-8") as f:
        for ex in splits[name]:
            f.write(json.dumps(ex, ensure_ascii=False) + "\n")
    print(f"Saved {path} ({len(splits[name])})")


Saved ft_out_offsides/train.jsonl (7149)
Saved ft_out_offsides/val.jsonl (1552)
Saved ft_out_offsides/test.jsonl (3431)
